# 00 — 语料形状

读 `s0_manifest.jsonl`，回答这 4083 份 session 是什么。只切 s0 已产好的表，不补算。

## 0. 产物版本

In [ ]:
import nbio
import pandas as pd
import plotly.express as px

pd.set_option("display.max_rows", 80)
nbio.banner()

s0–s2 与 s3、s4 的 git rev 不同是预期的：后两阶段后加，s0 未重跑。`limit` 全为 `None` 即全语料口径。

In [ ]:
s0 = nbio.load("s0")
summ = nbio.summary("s0")
s0.shape

## 1. 纳入与排除

`exclude` 是理由列表，一份 session 可同时撞上多条。

In [ ]:
print(f"总计 {len(s0)}，纳入 {s0.included.sum()}，排除 {(~s0.included).sum()}")
print()
print(s0[s0.included].family.value_counts().to_string())
print()
print(f"覆盖论文 {s0[s0.included].paper_id.nunique()} 篇，"
      f"纳入但无 paper_id {s0[s0.included].paper_id.isna().sum()} 份")

### 排除理由不互斥

`s0_summary.json` 的 `excluded_by_reason` 按理由计数，加总超过被排除的 session 数。

In [ ]:
exc = s0[~s0.included].copy()
exc["combo"] = exc.exclude.apply(lambda xs: " + ".join(sorted(xs)))
combo = exc.combo.value_counts().rename("会话数").to_frame()
combo["占被排除"] = (combo.会话数 / len(exc)).map("{:.1%}".format)

print(f"被排除 {len(exc)} 份，理由提及 {exc.exclude.apply(len).sum()} 次")
print()
print(combo.to_string())
print()
print("summary 里的按理由计数：", summ["excluded_by_reason"])

单独只是 `operator_chat` 的仅 2 份，其余 10 份同时满足别的条件。"12 份操作者交互"是理由计数，不能读作"12 份人工会话"。

## 2. 流产率

`aborted`（`step.begin` 发了、`step.end` 没到）在 s0 里是观测量而非噪声，故单列。

In [ ]:
ab = pd.DataFrame(summ["abort_rate_by_family"]).T
ab["rate"] = ab["rate"].map("{:.2%}".format)
ab

## 3. 语料的时间结构

`s0_summary.json` 把 `sysprompt_chars` 报成扁平直方图，丢掉了时间轴。接上 `created_at` 后可见它沿时间换代。成因见 `01_session_classes.ipynb`。

In [ ]:
s0["day"] = pd.to_datetime(s0.created_at).dt.tz_convert("Asia/Shanghai").dt.date
by_day = s0.groupby("day").agg(
    n=("sid", "size"),
    n_sysprompt=("sysprompt_chars", "nunique"),
    sysprompt=("sysprompt_chars", lambda s: sorted(s.dropna().unique())),
    n_repair=("family", lambda s: (s == "repair").sum()),
)
by_day

In [ ]:
fig = px.scatter(
    s0.dropna(subset=["sysprompt_chars"]),
    x="created_at", y="sysprompt_chars", color="family",
    title="system prompt 长度沿时间换代（每点一份 session）",
    labels={"created_at": "会话创建时间", "sysprompt_chars": "systemPrompt 字符数"},
    height=420, opacity=0.5,
)
fig.update_traces(marker_size=4)
fig.show()

- repair 仅出现在头两天（06-12、06-15）。22.2% 的流产率是语料早期的性质，不是 repair 任务的稳态性质。
- 跨 run 前缀分歧有相当部分来自时间推移。日内 system prompt 近乎常数，按天分组调度是低成本的改进。

## 4. 复算

文档第 1 节的数字是脚本算完人手抄进散文的。下面逐条复算。

In [ ]:
checks = {
    "总 session":      (len(s0), 4083),
    "纳入":            (int(s0.included.sum()), 3999),
    "extract":         (int((s0.included & (s0.family == "extract")).sum()), 3762),
    "repair":          (int((s0.included & (s0.family == "repair")).sum()), 237),
    "覆盖论文":        (int(s0[s0.included].paper_id.nunique()), 3321),
    "排除":            (int((~s0.included).sum()), 84),
    "aborted 理由":    (int(s0.exclude.apply(lambda xs: "aborted" in xs).sum()), 73),
    "operator_chat":   (int(s0.exclude.apply(lambda xs: "operator_chat" in xs).sum()), 12),
    "no_steps":        (int(s0.exclude.apply(lambda xs: "no_steps" in xs).sum()), 5),
    "multi_turn":      (int(s0.exclude.apply(lambda xs: "multi_turn" in xs).sum()), 4),
}
bad = {k: v for k, v in checks.items() if v[0] != v[1]}
for k, (got, want) in checks.items():
    print(f"{'ok ' if got == want else '✗  '} {k:16} 复算 {got:>6}  文档 {want:>6}")
assert not bad, f"文档与产物不一致：{bad}"